# 3. Analysis

Goal: recompute, from the raw samples, the values of the table at the end of `001MoDe_R1.marker.csv`.

| Column | Meaning |
|---|---|
| nLaps | number of laps around the circle (negative = clockwise) |
| Re | effective radius (mean distance to the center), pixel |
| Te | effective tolerance (spread of the distance to the center), pixel |
| error | % of the angle travelled outside the target |
| MT/lap | movement time per lap, s/lap |
| IDe/lap | effective index of difficulty, bit/lap |
| Be | effective tolerance divided by task tolerance (no unit) |
| IPe | effective index of performance, bit/s |

The definitions follow the documentation of MouseReMoCo (class `PerformanceAtTask`, https://github.com/DenisMot/mouseReMoCo/blob/master/src/fr/lgi2p/digit/ui/PerformanceAtTask.java), which describes how the effective performance is computed from the samples.

Each step below explains the idea first, then gives the code.

In [13]:
# imports 
import numpy as np
import matplotlib.pyplot as plt
import os

# if the working directory is notebooks/, move to the project root
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

from src import io_mouse as mm

DATA_FILE   = "data/001MoDe_R1.csv"
MARKER_FILE = "data/001MoDe_R1.marker.csv"

## Step 1: Load the data and the records
We reuse `io_mouse` (notebook 01). `x` and `y` are already centered on the task center, with y pointing up.

In [14]:
header = mm.read_header(DATA_FILE)
data = mm.read_data(DATA_FILE, header)
markers, ref_table = mm.read_markers(MARKER_FILE)
records = mm.get_records(markers)

R_TASK = header["taskRadius"]      # 209.5 px, radius of the ideal circle
T_TASK = header["taskTolerance"]   # 47 px, width of the target ring
print("Records:", len(records))

Records: 5


## Step 2: Select the samples of one record
A record goes from a `DoRecord` marker to the next `DoPause` marker.

MouseReMoCo says: *"count starts when the move has reached the path"*. At the beginning of a record the cursor is often far from the circle; these moves are not part of the circular movement. So we start at the first sample where the cursor is inside the target(`mouseInTarget = 1`).

In [15]:
def select_record(data, t_start, t_end):
    """Return t_ms, x, y, inside of one record, starting at the first sample inside the target."""
    in_record = (data["t_ms"] >= t_start) & (data["t_ms"] <= t_end)
    t = data["t_ms"][in_record]
    x = data["x"][in_record]
    y = data["y"][in_record]
    inside = data["inside"][in_record]
    first = np.argmax(inside)          # index of the first True
    return t[first:], x[first:], y[first:], inside[first:]

## Step 3: Moves between two samples
The software works on moves; a move goes from sample A to the next sample B. For each move we compute the angular distance (the change of the phase angle `arctan2(y, x)` of the cursor around the center), in radian.

The angle jumps from +pi to -pi each time the cursor crosses the left side of the circle. We bring each difference back into [-pi, pi] to remove this jump (with a modulo, no loop needed).

In [16]:
def angular_moves(x, y):
    """Signed angle (radian) travelled by each move, from sample A to the next sample B."""
    angle = np.arctan2(y, x)
    delta = np.diff(angle)
    return (delta + np.pi) % (2 * np.pi) - np.pi

## Step 4: Number of laps (nLaps)
Sum of all the angular moves divided by 2 pi. The sign gives the direction. if it is negative = clockwise (y points up).

In [17]:
def n_laps(x, y):
    return np.sum(angular_moves(x, y)) / (2 * np.pi)

## Step 5: Effective radius (Re) and effective tolerance (Te)
The distance to the center is `r = sqrt(x^2 + y^2)`.

- **Re** is the mean of `r`: the radius the participant really used. The task asks for 209.5 px.
- **Te** is `sqrt(2 pi e) * standard deviation of r` (about 4.133 standard deviations, as in the effective width of Fitts' law). This is what the Java code computes (`effectiveTolerance`). A larger Te means a less precise movement. We use the sample standard deviation (`ddof=1`), which matches the reference table slightly better.

In [18]:
def effective_radius(x, y):
    return np.mean(np.hypot(x, y))

def effective_tolerance(x, y):
    return np.sqrt(2 * np.pi * np.e) * np.std(np.hypot(x, y), ddof=1)

## Step 6: Error
MouseReMoCo counts the error with the angular distance, not with the number of samples: the angle travelled by the moves that end outside the target (`angleOutside`), divided by the total angle travelled (`angleTotal`). Both are signed sums, as in the Java code, so the sign cancels in the ratio.

This definition reproduces the error of the reference table for the 5 records.

In [19]:
def error_percent(x, y, inside):
    moves = angular_moves(x, y)
    ends_outside = ~inside[1:]          # sample B of each move
    return 100 * np.sum(moves[ends_outside]) / np.sum(moves)

## Step 7: Movement time per lap (MT/lap)
Duration of the analysed part of the record divided by the number of laps (absolute value).

In [20]:
def movement_time_per_lap(t_ms, laps):
    duration = (t_ms[-1] - t_ms[0]) / 1000   # seconds
    return duration / abs(laps)

## Step 8: Index of difficulty (IDe/lap), Be and index of performance (IPe)
- **IDe/lap** = `2 pi Re / Te` (bit/lap). The circumference divided by the tolerance is how many "target widths" fit in one lap. 
  In theory, 2 pi * 209.5 / 47 = 28.0, which is the `indexOfDifficulty` of the header.
- **Be** = `Te / task tolerance`. Be = 1 means exactly as precise as asked, Be > 1 less precise, Be < 1 more precise.
- **IPe** = `IDe / MT` (bit/s): the information rate, in bit per second (as in the SOFMER 2021 poster).

In [21]:
def ide_per_lap(Re, Te):
    return 2 * np.pi * Re / Te

def be(Te, tolerance):
    return Te / tolerance

def ipe(ide, mt):
    return ide / mt

## Step 9: Apply everything to each record

In [22]:
results = []
for t_start, t_end in records:
    t, x, y, inside = select_record(data, t_start, t_end)
    laps = n_laps(x, y)
    Re = effective_radius(x, y)
    Te = effective_tolerance(x, y)
    mt = movement_time_per_lap(t, laps)
    ide = ide_per_lap(Re, Te)
    results.append((laps, Re, Te, error_percent(x, y, inside), mt, ide, be(Te, T_TASK), ipe(ide, mt)))

results = np.array(results)

## Step 10: Compare with the reference table
Computed values on the first line of each record, the reference line from the marker file just under it.

In [23]:
names = ["nLaps", "Re", "Te", "error", "MT/lap", "IDe/lap", "Be", "IPe"]
print("Rec   " + " ".join(f"{n:>8}" for n in names))

def parse_ref(line):
    parts = [p.strip() for p in line.split(",")]
    return parts[0], [float(p) if p not in ("", "NaN") else np.nan for p in parts[1:9]]

ref_rows = [parse_ref(l) for l in ref_table[3:8]]   # Rec001 to Rec005

for i, row in enumerate(results):
    print(f"{i+1:<5} " + " ".join(f"{v:8.2f}" for v in row))
    print(f"ref   " + " ".join(f"{v:8.2f}" for v in ref_rows[i][1]))

Rec      nLaps       Re       Te    error   MT/lap  IDe/lap       Be      IPe
1       -10.84   213.86    91.80     3.90     1.84    14.64     1.95     7.94
ref     -10.85   213.87    91.79     3.90     1.84    14.64     1.95     7.94
2       -11.97   209.85    34.46     0.74     1.59    38.27     0.73    24.04
ref     -11.97   209.85    34.46     0.74     1.59    38.27     0.73    24.04
3       -13.26   209.43    42.42     2.81     1.45    31.02     0.90    21.40
ref     -13.26   209.43    42.42     2.81     1.45    31.02     0.90    21.40
4       -12.83   209.46    37.14     0.13     1.49    35.43     0.79    23.72
ref     -12.83   209.46    37.14     0.13     1.49    35.43     0.79    23.72
5       -14.74   210.89    43.88     2.84     1.30    30.20     0.93    23.20
ref     -14.74   210.89    43.88     2.84     1.30    30.20     0.93    23.20


## Step 11: Check that it matches
For each column except `error`, the difference with the reference must stay small (the reference is rounded to 2 decimals).

In [24]:
ref_values = np.array([r[1] for r in ref_rows])
diff = np.abs(results - ref_values)
for j, name in enumerate(names):
    print(f"{name:>8}: max difference = {diff[:, j].max():.3f}")

   nLaps: max difference = 0.012
      Re: max difference = 0.007
      Te: max difference = 0.006
   error: max difference = 0.005
  MT/lap: max difference = 0.004
 IDe/lap: max difference = 0.004
      Be: max difference = 0.004
     IPe: max difference = 0.005
